# Guide 21: OpenDSS Power Flow Analysis

Distribution power flow calculates voltage at every bus and current through every element
in the network. This guide walks through the SP&L model results.

**What you will learn:**
- How to interpret snapshot power flow results
- Voltage profile analysis with ANSI limits
- Line and transformer loading assessment
- System loss breakdown


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

RESULTS = '../sisyphean-power-and-light/network/results'


## Circuit Summary


In [ ]:
with open(f'{RESULTS}/summary.json') as f:
    summary = json.load(f)

for k, v in summary.items():
    print(f'{k:25s}: {v}')


## Voltage Profile

ANSI C84.1 Range A requires service voltage between 0.95 and 1.05 per-unit.
Buses outside this range indicate potential voltage regulation issues.


In [ ]:
voltages = pd.read_parquet(f'{RESULTS}/bus_voltages.parquet')

fig, ax = plt.subplots(figsize=(10, 5))
v = voltages['voltage_pu'][voltages['voltage_pu'] > 0.5]
ax.hist(v, bins=50, color='#1C4855', edgecolor='white', alpha=0.8)
ax.axvline(0.95, color='#E74C3C', linestyle='--', label='ANSI lower (0.95)')
ax.axvline(1.05, color='#E74C3C', linestyle='--', label='ANSI upper (1.05)')
ax.set_xlabel('Voltage (per-unit)')
ax.set_ylabel('Number of Buses')
ax.set_title('Bus Voltage Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Buses below 0.95 pu: {(v < 0.95).sum()}')
print(f'Buses above 1.05 pu: {(v > 1.05).sum()}')
print(f'Mean voltage: {v.mean():.4f} pu')


## Line Loading

Loading percentage shows how close each line is to its thermal rating.
Lines above 80% are candidates for upgrade or load transfer.


In [ ]:
lines = pd.read_parquet(f'{RESULTS}/line_flows.parquet')
top_lines = lines.nlargest(20, 'loading_pct')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top_lines)), top_lines['loading_pct'], color='#5FCCDB')
ax.set_yticks(range(len(top_lines)))
ax.set_yticklabels(top_lines['line_name'], fontsize=8)
ax.set_xlabel('Loading (%)')
ax.set_title('Top 20 Loaded Lines')
ax.axvline(100, color='#E74C3C', linestyle='--', label='Thermal limit')
ax.legend()
plt.tight_layout()
plt.show()


## Transformer Loading


In [ ]:
xfmrs = pd.read_parquet(f'{RESULTS}/transformer_loading.parquet')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(xfmrs['loading_pct'], bins=40, color='#1C4855', edgecolor='white', alpha=0.8)
ax.axvline(80, color='#E7A33E', linestyle='--', label='80% threshold')
ax.set_xlabel('Loading (%)')
ax.set_ylabel('Number of Transformers')
ax.set_title('Transformer Loading Distribution')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean loading: {xfmrs["loading_pct"].mean():.1f}%')
print(f'Max loading: {xfmrs["loading_pct"].max():.1f}%')
print(f'Transformers above 80%: {(xfmrs["loading_pct"] > 80).sum()}')


## System Losses

Losses in distribution systems are dominated by I²R heating in conductors.
Lines typically account for 60-80% of total losses.


In [ ]:
losses = pd.read_csv(f'{RESULTS}/system_losses.csv')
by_type = losses.groupby('element_type')['kw_loss'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

by_type.plot.bar(ax=axes[0], color='#1C4855')
axes[0].set_ylabel('Total Losses (kW)')
axes[0].set_title('Losses by Element Type')
axes[0].tick_params(axis='x', rotation=0)

top20 = losses.nlargest(20, 'kw_loss')
axes[1].barh(range(len(top20)), top20['kw_loss'], color='#5FCCDB')
axes[1].set_yticks(range(len(top20)))
axes[1].set_yticklabels(top20['element_name'], fontsize=7)
axes[1].set_xlabel('Loss (kW)')
axes[1].set_title('Top 20 Loss Contributors')

plt.tight_layout()
plt.show()

total = losses['kw_loss'].sum()
print(f'Total losses: {total:.1f} kW')
for t, v in by_type.items():
    print(f'  {t}: {v:.1f} kW ({v/total*100:.1f}%)')


## Summary

Key takeaways from the SP&L power flow analysis:

1. **Voltage**: Most buses are within ANSI Range A. Buses at feeder endpoints show the most voltage drop.
2. **Loading**: Transformers average ~20% loading with headroom for DER growth.
3. **Losses**: Lines dominate system losses. The top 20 elements account for a disproportionate share.
4. **Next steps**: See Guide 22 for hosting capacity analysis and Guide 23 for loss reduction strategies.
